In [1]:
import numpy as np
from scipy.linalg import sqrtm

In [2]:
h = 0.1 #time step
n = 600 #number of time steps
tf = n*h #final time

rng = np.random.default_rng(1)

In [3]:
#Sensor noise

######## Sun Sensor ##########
m_sun = 1

#Inertial sun vector (arbitrary direction)
r_sun_N = np.array([[1.0],
                [0.0],
                [0.0]])

#Sun sensor error model
scale_std_sun = 2e-3 #0.2% scale-factor error
misalign_std_sun = 1e-3 #cross-axis misalignment

M_sun = np.eye(3)
M_sun += np.diag(scale_std_sun * rng.standard_normal(3))
Mis_sun = misalign_std_sun * rng.standard_normal((3,3))
np.fill_diagonal(Mis_sun, 0.0)
M_sun += Mis_sun

b_sun = np.array([0.002, -0.001, 0.001]) #constant bias

#Additive noise
sigma_sun = 40 #arcseconds
sigma_sun = sigma_sun / 180.0 * np.pi #convert arcseconds to radians
W_sun = (sigma_sun**2) * np.eye(3)

#########   Star tracker  ##############
m_st = 4
sigma_st = 1.0 #arcseconds
sigma_st = sigma_st * 1/3600.0 / 180.0 * np.pi #convert arcseconds to radians
W_st = (sigma_st**2) * np.eye(3)


############   Gyro   #############
scale_std_gyro = 5e-4 #gyro scale-factor error
misalign_std_gyro = 1e-3 #gyro misalignment error

Mgyro = np.eye(3)
Mgyro += np.diag(scale_std_gyro * rng.standard_normal(3))
Mis_gyro = misalign_std_gyro * rng.standard_normal((3,3))
np.fill_diagonal(Mis_gyro, 0.0)
Mgyro += Mis_gyro

sigma_gyro = np.deg2rad(0.02) #gyro measurement noise (rad/s)
Wgyro = (sigma_gyro**2) * np.eye(3)

#Bias?
b0 = np.deg2rad(np.array([0.05, -0.03, 0.02])) #initial bias
sigma_bias_rw = np.deg2rad(0.001) #bias random walk
Wbias = (sigma_bias_rw**2) * h * np.eye(3)

In [4]:
#Generate noisy vector (sun sensor) measurements
ytraj_sun = np.zeros((3*m_sun, n))
sqrtW_sun = sqrtm(W_sun)

for k in range(n):
    Qk = Q(xtraj[0:4, k])
    yk = np.zeros((3, m_sun))

    for ell in range(m_sun):
        yideal = Qk.T @ r_sun_N[:, ell]
        wk = sqrtW_sun @ np.random.randn(3)

        y = M_sun @ yideal + b_sun + wk

        # normalize because the filter uses this as a direction measurement
        y = y / np.linalg.norm(y)

        yk[:, ell] = y

    ytraj_sun[:, k] = yk.reshape(3*m_sun, order='F')

NameError: name 'Q' is not defined

In [ ]:
#Generate measurements

# Generate noisy star-tracker measurements
ytraj_st = np.zeros((4*m_st, n))
sqrtW_st = sqrtm(W_st)

for k in range(n):
    qk = xtraj[0:4, k]
    yk = np.zeros((4, m_st))

    wk = (sqrtW_st @ np.random.randn(3*m_st)).reshape((3, m_st), order='F')

    for ell in range(m_st):
        yk[:, ell] = L(qk) @ expq(wk[:, ell])

    ytraj_st[:, k] = yk.reshape(4*m_st, order='F')


# Generate noisy gyro measurements
ygyro = np.zeros((3, n))
btraj = np.zeros((3, n))

#initial gyro bias
btraj[:, 0] = b0

sqrtWgyro = sqrtm(Wgyro)
sqrtWbias = sqrtm(Wbias)

for k in range(n):
    wk = sqrtWgyro @ np.random.randn(3)

    if k > 0:
        vk = sqrtWbias @ np.random.randn(3)
        btraj[:, k] = btraj[:, k-1] + vk

    omega_true = xtraj[4:7, k]
    ygyro[:, k] = Mgyro @ omega_true + btraj[:, k] + wk

In [ ]:
print("M_sun =\n", M_sun)
print("\nb_sun =", b_sun)
print("\nW_sun =\n", W_sun)

print("\nMgyro =\n", Mgyro)
print("\nWgyro =\n", Wgyro)
print("\nWbias =\n", Wbias)
print("\nW_st =\n", W_st)